In [1]:
!pip install -q rutube-downloader google-genai opencv-python-headless
!apt-get -y install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 2.5 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [2]:
from rutube import Rutube
print("rutube-downloader OK")

rutube-downloader OK


In [3]:
# ============================================================
# TRON: Rutube → Кадры → Gemma 4 | Google Colab Pipeline
# ============================================================
# Установка зависимостей (запустить один раз):
# !pip install rutube-downloader google-genai opencv-python-headless
# !apt-get install -y ffmpeg

import os
import cv2
import time
import requests
import getpass
from pathlib import Path
from google import genai
from google.genai import types
from rutube import Rutube

# ── 1. НАСТРОЙКА ──────────────────────────────────────────────

api_key = getpass.getpass("Gemma API Key: ")
client = genai.Client(api_key=api_key)

WORK_DIR = Path("/content/tron")
WORK_DIR.mkdir(exist_ok=True)

# Запросы для поиска — чем конкретнее, тем лучше
SEARCH_QUERIES = [
    "кошка поведение дома",
    "собака тревога беспокойство",
    "животные странное поведение",
    "кот пугается",
]

# ── 2. ПОИСК ВИДЕО НА РУТУБЕ ──────────────────────────────────

def search_rutube(query: str, max_results: int = 5) -> list[dict]:
    """Ищет видео на Рутубе через публичный API."""
    url = "https://rutube.ru/api/video/"
    params = {
        "query": query,
        "format": "json",
        "page": 1,
        "page_size": max_results,
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        results = resp.json().get("results", [])
        videos = []
        for v in results:
            videos.append({
                "title": v.get("title", ""),
                "video_url": v.get("video_url", ""),
                "thumbnail_url": v.get("thumbnail_url", ""),
                "duration": v.get("duration", 0),
            })
        return videos
    except Exception as e:
        print(f"  [!] Ошибка поиска '{query}': {e}")
        return []


def collect_video_urls(queries: list[str], max_per_query: int = 3) -> list[dict]:
    """Собирает URL видео по всем запросам, убирает дубликаты."""
    seen = set()
    all_videos = []
    for q in queries:
        print(f"🔍 Поиск: '{q}'")
        videos = search_rutube(q, max_per_query)
        for v in videos:
            url = v["video_url"]
            if url and url not in seen:
                seen.add(url)
                all_videos.append(v)
                print(f"   + {v['title'][:60]} ({v['duration']}s)")
        time.sleep(1)  # не спамим API
    print(f"\n✅ Найдено уникальных видео: {len(all_videos)}")
    return all_videos


# ── 3. СКАЧИВАНИЕ ВИДЕО ───────────────────────────────────────

def download_video(video_url: str, out_dir: Path) -> Path | None:
    """Скачивает видео с Рутуба, возвращает путь к файлу."""
    safe_name = video_url.rstrip("/").split("/")[-1]
    out_path = out_dir / f"{safe_name}.mp4"

    if out_path.exists():
        print(f"  [=] Уже скачано: {out_path.name}")
        return out_path

    try:
        rt = Rutube(video_url)
        # Берём худшее качество — файл меньше, Gemma всё равно видит
        video = rt.get_worst()
        if video is None:
            print(f"  [!] Нет доступных потоков: {video_url}")
            return None
        with open(out_path, "wb") as f:
            video.download(stream=f)
        print(f"  [✓] Скачано: {out_path.name}")
        return out_path
    except Exception as e:
        print(f"  [!] Ошибка скачивания {video_url}: {e}")
        return None


# ── 4. НАРЕЗКА КАДРОВ ─────────────────────────────────────────

def extract_frames(video_path: Path, out_dir: Path,
                   every_n_seconds: int = 5, max_frames: int = 10) -> list[Path]:
    """
    Нарезает видео на кадры каждые N секунд.
    Возвращает список путей к JPEG-файлам.
    """
    frames_dir = out_dir / video_path.stem
    frames_dir.mkdir(exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"  [!] Не удалось открыть видео: {video_path}")
        return []

    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    frame_interval = int(fps * every_n_seconds)
    frame_count = 0
    saved = []

    while len(saved) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % frame_interval == 0:
            frame_path = frames_dir / f"frame_{frame_count:06d}.jpg"
            cv2.imwrite(str(frame_path), frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
            saved.append(frame_path)
        frame_count += 1

    cap.release()
    print(f"  [✓] Кадров извлечено: {len(saved)}")
    return saved


# ── 5. АНАЛИЗ ЧЕРЕЗ GEMMA 4 ───────────────────────────────────

PROMPT = """Ты — ИИ-система мониторинга поведения животных для раннего обнаружения аномалий перед землетрясениями (проект TRON).

Проанализируй кадр и определи поведение животного.

Ответь строго в формате:
СТАТУС: [NORMAL / SUSPICIOUS / ANOMALY]
ЖИВОТНОЕ: [что за животное, если видно]
ПРИЗНАКИ: [конкретные признаки поведения которые ты видишь]
УВЕРЕННОСТЬ: [HIGH / MEDIUM / LOW]
"""

def analyze_frame(frame_path: Path) -> dict:
    """Анализирует один кадр через Gemma 4."""
    img_bytes = frame_path.read_bytes()
    try:
        response = client.models.generate_content(
            model="gemma-4-31b-it",
            contents=[
                types.Part.from_bytes(data=img_bytes, mime_type="image/jpeg"),
                PROMPT,
            ]
        )
        text = response.text
        # Простой парсинг ответа
        result = {"raw": text, "frame": str(frame_path)}
        for line in text.splitlines():
            if line.startswith("СТАТУС:"):
                result["status"] = line.split(":", 1)[1].strip()
            elif line.startswith("ЖИВОТНОЕ:"):
                result["animal"] = line.split(":", 1)[1].strip()
            elif line.startswith("ПРИЗНАКИ:"):
                result["signs"] = line.split(":", 1)[1].strip()
            elif line.startswith("УВЕРЕННОСТЬ:"):
                result["confidence"] = line.split(":", 1)[1].strip()
        return result
    except Exception as e:
        return {"frame": str(frame_path), "error": str(e)}


def analyze_video(frames: list[Path], video_title: str) -> dict:
    """Анализирует все кадры видео, возвращает итоговую оценку."""
    print(f"\n  🧠 Анализ: {video_title[:50]}")
    results = []
    for frame in frames:
        r = analyze_frame(frame)
        status = r.get("status", "?")
        print(f"     {frame.name}: {status} | {r.get('signs', r.get('error', ''))[:60]}")
        results.append(r)
        time.sleep(0.5)  # небольшая пауза между запросами

    # Итоговая оценка: worst-case по видео
    statuses = [r.get("status", "") for r in results]
    if "ANOMALY" in statuses:
        overall = "ANOMALY"
    elif "SUSPICIOUS" in statuses:
        overall = "SUSPICIOUS"
    else:
        overall = "NORMAL"

    anomaly_count = statuses.count("ANOMALY")
    suspicious_count = statuses.count("SUSPICIOUS")

    return {
        "title": video_title,
        "overall": overall,
        "frames_analyzed": len(results),
        "anomaly_frames": anomaly_count,
        "suspicious_frames": suspicious_count,
        "frame_results": results,
    }


# ── 6. ГЛАВНЫЙ ЦИКЛ ───────────────────────────────────────────

def run_pipeline(max_videos: int = 5):
    print("=" * 60)
    print("TRON | Rutube → Gemma 4 Pipeline")
    print("=" * 60)

    # Шаг 1: поиск
    all_videos = collect_video_urls(SEARCH_QUERIES, max_per_query=3)
    all_videos = all_videos[:max_videos]  # лимит для теста

    video_dir = WORK_DIR / "videos"
    video_dir.mkdir(exist_ok=True)

    pipeline_results = []

    for i, v in enumerate(all_videos, 1):
        print(f"\n[{i}/{len(all_videos)}] {v['title'][:60]}")
        print(f"  URL: {v['video_url']}")

        # Шаг 2: скачивание
        video_path = download_video(v["video_url"], video_dir)
        if not video_path:
            continue

        # Шаг 3: кадры
        frames = extract_frames(
            video_path,
            out_dir=WORK_DIR / "frames",
            every_n_seconds=5,
            max_frames=8,
        )
        if not frames:
            continue

        # Шаг 4: Gemma
        result = analyze_video(frames, v["title"])
        pipeline_results.append(result)

    # Итог
    print("\n" + "=" * 60)
    print("ИТОГИ АНАЛИЗА")
    print("=" * 60)
    for r in pipeline_results:
        emoji = {"NORMAL": "🟢", "SUSPICIOUS": "🟡", "ANOMALY": "🔴"}.get(r["overall"], "⚪")
        print(f"{emoji} {r['overall']:10s} | кадров: {r['frames_analyzed']} "
              f"(⚠️{r['suspicious_frames']} 🔴{r['anomaly_frames']}) | {r['title'][:45]}")

    return pipeline_results


# ── ЗАПУСК ────────────────────────────────────────────────────
if __name__ == "__main__":
    results = run_pipeline(max_videos=5)

Gemma API Key: ··········
TRON | Rutube → Gemma 4 Pipeline
🔍 Поиск: 'кошка поведение дома'
  [!] Ошибка поиска 'кошка поведение дома': 403 Client Error: Forbidden for url: https://rutube.ru/api/video/?query=%D0%BA%D0%BE%D1%88%D0%BA%D0%B0+%D0%BF%D0%BE%D0%B2%D0%B5%D0%B4%D0%B5%D0%BD%D0%B8%D0%B5+%D0%B4%D0%BE%D0%BC%D0%B0&format=json&page=1&page_size=3
🔍 Поиск: 'собака тревога беспокойство'
  [!] Ошибка поиска 'собака тревога беспокойство': 403 Client Error: Forbidden for url: https://rutube.ru/api/video/?query=%D1%81%D0%BE%D0%B1%D0%B0%D0%BA%D0%B0+%D1%82%D1%80%D0%B5%D0%B2%D0%BE%D0%B3%D0%B0+%D0%B1%D0%B5%D1%81%D0%BF%D0%BE%D0%BA%D0%BE%D0%B9%D1%81%D1%82%D0%B2%D0%BE&format=json&page=1&page_size=3
🔍 Поиск: 'животные странное поведение'
  [!] Ошибка поиска 'животные странное поведение': 403 Client Error: Forbidden for url: https://rutube.ru/api/video/?query=%D0%B6%D0%B8%D0%B2%D0%BE%D1%82%D0%BD%D1%8B%D0%B5+%D1%81%D1%82%D1%80%D0%B0%D0%BD%D0%BD%D0%BE%D0%B5+%D0%BF%D0%BE%D0%B2%D0%B5%D0%B4%D0%B5%D0%BD%D0

In [4]:
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://rutube.ru/",
    "Accept": "application/json, text/plain, */*",
}
resp = requests.get(
    "https://rutube.ru/api/video/",
    params={"query": "кошка", "format": "json", "page": 1, "page_size": 3},
    headers=headers,
    timeout=10
)
print(resp.status_code)
print(resp.json().get("results", [{}])[0].get("title", "пусто"))import requests

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://rutube.ru/",
    "Accept": "application/json, text/plain, */*",
}
resp = requests.get(
    "https://rutube.ru/api/video/",
    params={"query": "кошка", "format": "json", "page": 1, "page_size": 3},
    headers=headers,
    timeout=10
)
print(resp.status_code)
print(resp.json().get("results", [{}])[0].get("title", "пусто"))

SyntaxError: invalid syntax (3227727774.py, line 15)

In [5]:
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://rutube.ru/",
    "Accept": "application/json, text/plain, */*",
}
resp = requests.get(
    "https://rutube.ru/api/video/",
    params={"query": "кошка", "format": "json", "page": 1, "page_size": 3},
    headers=headers,
    timeout=10
)
print(resp.status_code)
print(resp.json().get("results", [{}])[0].get("title", "пусто"))

401
пусто


In [6]:
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://rutube.ru/",
    "Accept": "application/json, text/plain, */*",
}

resp = requests.get(
    "https://rutube.ru/api/search/video/",
    params={"query": "кошка", "page": 1, "page_size": 3},
    headers=headers,
    timeout=10
)
print(resp.status_code)
print(resp.text[:500])

200
{"count":0,"has_next":false,"next":null,"previous":null,"current_page":1,"results":[]}



In [7]:
resp = requests.get(
    "https://rutube.ru/api/search/video/",
    params={"query": "кошка", "page": 1, "page_size": 3, "format": "json"},
    headers=headers,
    timeout=10
)
print(resp.status_code)
print(resp.text[:1000])

200
{"count":0,"has_next":false,"next":null,"previous":null,"current_page":1,"results":[]}



In [8]:
resp = requests.get(
    "https://rutube.ru/api/video/?query=кошка",
    headers=headers,
    timeout=10
)
print(resp.status_code)
print(resp.text[:1000])

401
{"detail": "\u0423\u0447\u0435\u0442\u043d\u044b\u0435 \u0434\u0430\u043d\u043d\u044b\u0435 \u043d\u0435 \u0431\u044b\u043b\u0438 \u043f\u0440\u0435\u0434\u043e\u0441\u0442\u0430\u0432\u043b\u0435\u043d\u044b."}


In [9]:
resp = requests.get(
    "https://rutube.ru/api/search/?query=кошка&page=1",
    headers=headers,
    timeout=10
)
print(resp.status_code)
print(resp.text[:1000])

404

<!doctype html>
<html lang="en">
<head>
  <title>Not Found</title>
</head>
<body>
  <h1>Not Found</h1><p>The requested resource was not found on this server.</p>
</body>
</html>



In [10]:
from rutube import Rutube

rt = Rutube("https://rutube.ru/video/9e3ab6e4d2ff8b62e68e24fd46e46944/")
print(rt.available_resolutions)

Exception: https://rutube.ru/video/9e3ab6e4d2ff8b62e68e24fd46e46944/ is unavailable

In [11]:
from rutube import Rutube

rt = Rutube("https://rutube.ru/video/ССЫЛКА/")
rt.get_worst().download("C:/tron/videos/")

Exception: https://rutube.ru/video/ССЫЛКА/ is unavailable

In [12]:
import requests

urls = [
    "https://rutube.ru/video/beec1a97b2d3edade5a5c5e0ec043320/",
    "https://rutube.ru/video/ff20e6dc34f0cab03364a88269b469a5/",
    "https://rutube.ru/video/dc77b68bdb516ac535a41f6fe1c19cf8/",
    "https://rutube.ru/video/316e7b42a8a7699c4bbda46ead6581df/",
    "https://rutube.ru/video/24b73e0da0f68cc2a0c7bb5958b50d80/",
]

for url in urls:
    resp = requests.get(url, headers=headers, timeout=10)
    print(resp.status_code, url)

200 https://rutube.ru/video/beec1a97b2d3edade5a5c5e0ec043320/
200 https://rutube.ru/video/ff20e6dc34f0cab03364a88269b469a5/
200 https://rutube.ru/video/dc77b68bdb516ac535a41f6fe1c19cf8/
200 https://rutube.ru/video/316e7b42a8a7699c4bbda46ead6581df/
200 https://rutube.ru/video/24b73e0da0f68cc2a0c7bb5958b50d80/


In [13]:
from rutube import Rutube

rt = Rutube("https://rutube.ru/video/beec1a97b2d3edade5a5c5e0ec043320/")
print(rt.available_resolutions)

Exception: https://rutube.ru/video/beec1a97b2d3edade5a5c5e0ec043320/ is unavailable

In [14]:
import requests

video_id = "beec1a97b2d3edade5a5c5e0ec043320"
resp = requests.get(
    f"https://rutube.ru/api/play/options/{video_id}/",
    headers=headers,
    timeout=10
)
print(resp.status_code)
print(resp.text[:1000])

200
{"acl_access":{"allowed":true,"err_code":null,"err_text":""},"advert":[],"appearance":{"color":"00A1E7","auto_start":true,"show_subscription":true,"show_author":true,"show_title":true,"show_hd":false,"show_logotype":false,"show_full_tab":true,"show_avatar":true,"show_top_panel":true,"show_share_button":true,"show_social_buttons":false,"show_endscreen":true,"show_related":true,"show_category_recommended":true,"show_user_recommended":true,"show_pg_rating":false,"available_pause_click":true,"available_title_click":false,"available_logo_click":false,"available_author_click":false,"available_related_click":true,"plugin_modifying":true,"forbid_html_controls":false,"show_adult_stub":true,"available_link_stub":true,"available_special_source":false,"advert_skin_type":null,"show_preloader":true,"enable_embed_logo":true,"cookies_mail":false,"forbid_html_fullscreen":false,"allow_auto_seek":true,"forbid_timeline_preview":false,"allow_push_subscribe":false,"enable_playlist":false,"show_logo_with

In [15]:
data = resp.json()
video_balancer = data.get("video_balancer", {})
print(video_balancer)

{'default': 'https://bl.rutube.ru/route/beec1a97b2d3edade5a5c5e0ec043320.m3u8?guids=46f827e8-d7ca-4291-8424-c37f90d963a9_1920x1080_4435316_D387649_B4236664A192000_F25A44100_avc1.640029_mp4a.40.2,6f97acb1-6ac2-4003-be90-9fe6e0a7ba45_1280x720_2213322_D387649_B2014618A192000_F25A44100_avc1.640029_mp4a.40.2,44c17922-041a-47fc-94d6-d38371a7915c_848x480_1228642_D387649_B1093921A128008_F25A44100_avc1.4d401f_mp4a.40.2,5efeb06f-9b88-4030-bbaa-2def02b4726a_640x360_1203269_D387649_B1134113A64020_F25A44100_avc1.42c01f_mp4a.40.2,a5a22c2b-81e0-4f8c-b041-97a21cd9e13a_424x240_652047_D387649_B582878A64020_F25A44100_avc1.42c01f_mp4a.40.2,6b9ffa51-2186-4715-9982-203a062cde18_256x144_325484_D387649_B256307A64020_F25A44100_avc1.42c01f_mp4a.40.2&sign=se2SZD0cy6NU5cuOgcEVbw&expire=1775450292&guarantee=6&scheme=https', 'm3u8': 'https://bl.rutube.ru/route/beec1a97b2d3edade5a5c5e0ec043320.m3u8?guids=46f827e8-d7ca-4291-8424-c37f90d963a9_1920x1080_4435316_D387649_B4236664A192000_F25A44100_avc1.640029_mp4a.40.2,6f

In [16]:
import subprocess
from pathlib import Path

m3u8_url = data["video_balancer"]["m3u8"]
out_path = "/content/tron/video1.mp4"
Path("/content/tron").mkdir(exist_ok=True)

subprocess.run([
    "ffmpeg", "-i", m3u8_url,
    "-c", "copy", "-t", "30",  # только первые 30 секунд
    out_path, "-y"
], capture_output=True)

print("Размер файла:", Path(out_path).stat().st_size, "байт")

Размер файла: 10794105 байт


In [17]:
import cv2

cap = cv2.VideoCapture("/content/tron/video1.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
frames_saved = []
frame_count = 0

Path("/content/tron/frames").mkdir(exist_ok=True)

while len(frames_saved) < 6:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_count % int(fps * 5) == 0:  # каждые 5 секунд
        path = f"/content/tron/frames/frame_{frame_count:06d}.jpg"
        cv2.imwrite(path, frame)
        frames_saved.append(path)
    frame_count += 1

cap.release()
print(f"Кадров извлечено: {len(frames_saved)}")

Кадров извлечено: 6


In [18]:
from google import genai
from google.genai import types

for frame_path in frames_saved:
    img_bytes = open(frame_path, "rb").read()
    response = client.models.generate_content(
        model="gemma-4-31b-it",
        contents=[
            types.Part.from_bytes(data=img_bytes, mime_type="image/jpeg"),
            "Опиши поведение животного. Оцени: NORMAL / SUSPICIOUS / ANOMALY. Укажи признаки."
        ]
    )
    print(f"{frame_path.split('/')[-1]}: {response.text[:200]}")
    print("---")

frame_000000.jpg: Поведение животного: Кот спокойно находится на руках у человека, его поза расслаблена, взгляд направлен в сторону. Признаков стресса, агрессии или возбуждения не наблюдается.

Оценка: NORMAL

Признаки
---
frame_000125.jpg: **Поведение животного:** Кошка (породы сфинкс) находится в руках у человека. Она выглядит напряженной, её тело сковано, голова отвернута в сторону от владельца, рот слегка приоткрыт (что может указыва
---
frame_000250.jpg: **Поведение животного:** Кот проявляет любопытство и внимание, глядя прямо в камеру. Он находится в спокойном состоянии, комфортно располагаясь рядом с человеком.

**Оценка:** NORMAL

**Признаки:** 
*
---
frame_000375.jpg: Поведение животного: Кошка (порода сфинкс) находится в руках у человека, проявляет любопытство или легкое удивление, глядя в камеру.

Оценка: **NORMAL**

Признаки:
1. **Спокойное состояние**: Животное
---
frame_000500.jpg: Поведение животного: Животное (вероятно, собака) совершает резкие, быстрые движения, похож

In [19]:
import requests
import subprocess
import cv2
from pathlib import Path
from google import genai
from google.genai import types

URLS = [
    "beec1a97b2d3edade5a5c5e0ec043320",
    "ff20e6dc34f0cab03364a88269b469a5",
    "dc77b68bdb516ac535a41f6fe1c19cf8",
    "316e7b42a8a7699c4bbda46ead6581df",
    "24b73e0da0f68cc2a0c7bb5958b50d80",
]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://rutube.ru/",
    "Accept": "application/json, text/plain, */*",
}

Path("/content/tron/videos").mkdir(parents=True, exist_ok=True)
Path("/content/tron/frames").mkdir(parents=True, exist_ok=True)

results = []

for video_id in URLS:
    print(f"\n{'='*50}")
    print(f"Видео: {video_id}")

    # Шаг 1: получаем m3u8
    resp = requests.get(
        f"https://rutube.ru/api/play/options/{video_id}/",
        headers=headers, timeout=10
    )
    m3u8_url = resp.json()["video_balancer"]["m3u8"]

    # Шаг 2: скачиваем первые 30 секунд
    video_path = f"/content/tron/videos/{video_id}.mp4"
    subprocess.run([
        "ffmpeg", "-i", m3u8_url,
        "-c", "copy", "-t", "30",
        video_path, "-y"
    ], capture_output=True)
    print(f"Скачано: {Path(video_path).stat().st_size // 1024} KB")

    # Шаг 3: нарезаем кадры
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    frames_saved = []
    frame_count = 0
    frames_dir = Path(f"/content/tron/frames/{video_id}")
    frames_dir.mkdir(exist_ok=True)

    while len(frames_saved) < 6:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % int(fps * 5) == 0:
            path = str(frames_dir / f"frame_{frame_count:06d}.jpg")
            cv2.imwrite(path, frame)
            frames_saved.append(path)
        frame_count += 1
    cap.release()
    print(f"Кадров: {len(frames_saved)}")

    # Шаг 4: Gemma
    statuses = []
    for frame_path in frames_saved:
        img_bytes = open(frame_path, "rb").read()
        response = client.models.generate_content(
            model="gemma-4-31b-it",
            contents=[
                types.Part.from_bytes(data=img_bytes, mime_type="image/jpeg"),
                "Оцени поведение животного: NORMAL / SUSPICIOUS / ANOMALY. Одним словом в первой строке, затем коротко признаки."
            ]
        )
        text = response.text.strip()
        status = "NORMAL"
        for s in ["ANOMALY", "SUSPICIOUS", "NORMAL"]:
            if s in text.upper():
                status = s
                break
        statuses.append(status)
        print(f"  {Path(frame_path).name}: {status}")

    # Итог по видео
    if "ANOMALY" in statuses:
        overall = "🔴 ANOMALY"
    elif "SUSPICIOUS" in statuses:
        overall = "🟡 SUSPICIOUS"
    else:
        overall = "🟢 NORMAL"

    results.append({"id": video_id, "overall": overall, "statuses": statuses})

print(f"\n{'='*50}")
print("ИТОГИ:")
for r in results:
    print(f"{r['overall']} | {r['id']}")


Видео: beec1a97b2d3edade5a5c5e0ec043320
Скачано: 10541 KB
Кадров: 6
  frame_000000.jpg: NORMAL
  frame_000125.jpg: NORMAL
  frame_000250.jpg: SUSPICIOUS
  frame_000375.jpg: ANOMALY
  frame_000500.jpg: NORMAL
  frame_000625.jpg: ANOMALY

Видео: ff20e6dc34f0cab03364a88269b469a5
Скачано: 1882 KB
Кадров: 6
  frame_000000.jpg: NORMAL
  frame_000125.jpg: NORMAL
  frame_000250.jpg: NORMAL
  frame_000375.jpg: NORMAL
  frame_000500.jpg: NORMAL
  frame_000625.jpg: NORMAL

Видео: dc77b68bdb516ac535a41f6fe1c19cf8
Скачано: 1638 KB
Кадров: 6
  frame_000000.jpg: NORMAL
  frame_000150.jpg: NORMAL
  frame_000300.jpg: NORMAL
  frame_000450.jpg: NORMAL
  frame_000600.jpg: NORMAL
  frame_000750.jpg: NORMAL

Видео: 316e7b42a8a7699c4bbda46ead6581df
Скачано: 10084 KB
Кадров: 6
  frame_000000.jpg: NORMAL
  frame_000149.jpg: NORMAL
  frame_000298.jpg: ANOMALY
  frame_000447.jpg: ANOMALY
  frame_000596.jpg: ANOMALY
  frame_000745.jpg: ANOMALY

Видео: 24b73e0da0f68cc2a0c7bb5958b50d80
Скачано: 5455 KB
Кадров: 6


In [20]:
import requests

test_id = "97bc8c7da2dbb472f937d71c060d9ff8"
resp = requests.get(
    f"https://rutube.ru/api/play/options/{test_id}/",
    headers=headers,
    timeout=10
)
print(resp.status_code)
print(resp.json().get("video_balancer", {}).get("m3u8", "нет")[:80])

200
https://bl.rutube.ru/route/97bc8c7da2dbb472f937d71c060d9ff8.m3u8?guids=fc97537e-


In [21]:
URLS2 = [
    "97bc8c7da2dbb472f937d71c060d9ff8",  # рыбки беспокоятся
    "9202a37094c88bc6de055ee9decf0251",  # просто рыбки
    "398399360ba36dc12230f2321b0ebfc6",  # кошка с котятами
    "d28d17606082b9bf719338e007b7f93d",  # кошка
    "a373489a111867d1f5ac7e0ebba5456c",  # слоны во время землетрясения
]

results2 = []

for video_id in URLS2:
    print(f"\n{'='*50}")
    print(f"Видео: {video_id}")

    resp = requests.get(
        f"https://rutube.ru/api/play/options/{video_id}/",
        headers=headers, timeout=10
    )
    m3u8_url = resp.json()["video_balancer"]["m3u8"]

    video_path = f"/content/tron/videos/{video_id}.mp4"
    subprocess.run([
        "ffmpeg", "-i", m3u8_url,
        "-c", "copy", "-t", "30",
        video_path, "-y"
    ], capture_output=True)
    print(f"Скачано: {Path(video_path).stat().st_size // 1024} KB")

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    frames_saved = []
    frame_count = 0
    frames_dir = Path(f"/content/tron/frames/{video_id}")
    frames_dir.mkdir(exist_ok=True)

    while len(frames_saved) < 6:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % int(fps * 5) == 0:
            path = str(frames_dir / f"frame_{frame_count:06d}.jpg")
            cv2.imwrite(path, frame)
            frames_saved.append(path)
        frame_count += 1
    cap.release()
    print(f"Кадров: {len(frames_saved)}")

    statuses = []
    for frame_path in frames_saved:
        img_bytes = open(frame_path, "rb").read()
        response = client.models.generate_content(
            model="gemma-4-31b-it",
            contents=[
                types.Part.from_bytes(data=img_bytes, mime_type="image/jpeg"),
                "Оцени поведение животного: NORMAL / SUSPICIOUS / ANOMALY. Одним словом в первой строке, затем коротко признаки."
            ]
        )
        text = response.text.strip()
        status = "NORMAL"
        for s in ["ANOMALY", "SUSPICIOUS", "NORMAL"]:
            if s in text.upper():
                status = s
                break
        statuses.append(status)
        print(f"  {Path(frame_path).name}: {status}")

    if "ANOMALY" in statuses:
        overall = "🔴 ANOMALY"
    elif "SUSPICIOUS" in statuses:
        overall = "🟡 SUSPICIOUS"
    else:
        overall = "🟢 NORMAL"

    results2.append({"id": video_id, "overall": overall, "statuses": statuses})

print(f"\n{'='*50}")
print("ИТОГИ:")
for r in results2:
    print(f"{r['overall']} | {r['id']}")


Видео: 97bc8c7da2dbb472f937d71c060d9ff8
Скачано: 26267 KB
Кадров: 6
  frame_000000.jpg: NORMAL
  frame_000150.jpg: NORMAL
  frame_000300.jpg: NORMAL
  frame_000450.jpg: NORMAL
  frame_000600.jpg: NORMAL
  frame_000750.jpg: NORMAL

Видео: 9202a37094c88bc6de055ee9decf0251
Скачано: 5670 KB
Кадров: 6
  frame_000000.jpg: NORMAL
  frame_000150.jpg: NORMAL
  frame_000300.jpg: NORMAL
  frame_000450.jpg: NORMAL
  frame_000600.jpg: ANOMALY
  frame_000750.jpg: NORMAL

Видео: 398399360ba36dc12230f2321b0ebfc6
Скачано: 18382 KB
Кадров: 6
  frame_000000.jpg: NORMAL
  frame_000300.jpg: NORMAL
  frame_000600.jpg: NORMAL
  frame_000900.jpg: NORMAL
  frame_001200.jpg: NORMAL
  frame_001500.jpg: NORMAL

Видео: d28d17606082b9bf719338e007b7f93d
Скачано: 1128 KB
Кадров: 6
  frame_000000.jpg: NORMAL
  frame_000149.jpg: NORMAL
  frame_000298.jpg: NORMAL
  frame_000447.jpg: NORMAL
  frame_000596.jpg: NORMAL
  frame_000745.jpg: NORMAL

Видео: a373489a111867d1f5ac7e0ebba5456c
Скачано: 3770 KB
Кадров: 6
  frame_0

In [23]:
video_id = "a373489a111867d1f5ac7e0ebba5456c"  # слоны
frames_dir = Path(f"/content/tron/frames/{video_id}")
frame_files = sorted(frames_dir.glob("*.jpg"))

contents = []
for i, fp in enumerate(frame_files):
    img_bytes = fp.read_bytes()
    contents.append(types.Part.from_bytes(data=img_bytes, mime_type="image/jpeg"))
    contents.append(f"Кадр {i+1}")

contents.append("""
Это последовательные кадры из одного видео с интервалом 5 секунд.
Проанализируй ДИНАМИКУ поведения животных от кадра к кадру.
Есть ли изменение поведения во времени?
Итоговая оценка: NORMAL / SUSPICIOUS / ANOMALY и почему.
""")

response = client.models.generate_content(
    model="gemma-4-31b-it",
    contents=contents
)
print(response.text)

**Анализ динамики поведения животных:**

1.  **Кадр 1 и 2:** Слоны находятся в разобщенном состоянии, распределены по вольеру. Они занимаются своими обычными делами (перемещаются, стоят).
2.  **Кадр 3:** Начинается изменение паттерна поведения. Слоны начинают стягиваться к центру вольера. Один из слонов слева активно движется к остальным.
3.  **Кадр 4:** Процесс сближения продолжается. Животные сокращают дистанцию между собой, формируя группу в центре и справа.
4.  **Кадр 5:** Слоны практически объединились в одну плотную группу. Наблюдается выраженное стремление к физическому контакту и взаимной защите.
5.  **Кадр 6:** Группа остается компактной. Поведение характеризуется высокой степенью сплоченности, что нетипично для обычного спокойного состояния животных в вольере.

**Изменение поведения во времени:**
Да, наблюдается четкая динамика: от **разобщенности** $\rightarrow$ **сближения** $\rightarrow$ **формирования защитного кластера (круга)**. Этот процесс произошел в течение коротког